# Calculate top/bottom 10 subjects' dissim ~ genmag, spread

20260124

To create the library, implement and test the function here.

Calculate genmag and spread then store in work/data/processed as hdf style.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = os.path.abspath("/home/jovyan/work")
print("Project root:", project_root)

if project_root not in sys.path:
    sys.path.append(project_root)

from funcs import core, viz

In [ ]:
d_path = "/home/jovyan/work/data/raw/Amy_dissimilarity.csv"
unique_words = ["red", "orange", "yellow", "green", "blue", "purple", "pink", "brown", "grey", "black", "happiness", "joy", "confidence", "calm", "boredom", "confusion", "anxiety", "fear", "sadness", "defeated", "anger", "envy", "disgust"]
n_sub = 1
raw = pd.read_csv(d_path, header=None)
raw.drop(columns=0, inplace=True)
raw

In [ ]:
# extract top and bottom 10 participants
raw_header = raw.iloc[0:2, :].copy()

raw_top = raw.iloc[:12, :].copy()
raw_bottom_ = raw.iloc[-10:, :].copy()
raw_bottom = pd.concat([raw_header, raw_bottom_])
raw_bottom

In [ ]:
# define t

t1 = np.logspace(-4, -1, 20, endpoint=False)   # 10^-4 ～ 10^-1
t2 = np.logspace(-1,  2, 500, endpoint=False)  # 10^-1 ～ 10^2
t3 = np.logspace(2,  4, 20, endpoint=False)    # 10^2 ～ 10^4
t4 = np.logspace(4, 8, 20, endpoint=True)

t_all = np.concatenate([t1, t2, t3, t4])

In [ ]:
genmag_top = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

spr_top = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

positive_definite_t_top = pd.DataFrame(
    data=pd.NA,
    index=np.arange(n_sub),
    columns=t_all,
    dtype="boolean"
)

genmag_bot = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

spr_bot = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

positive_definite_t_bot = pd.DataFrame(
    data=pd.NA,
    index=np.arange(n_sub),
    columns=t_all,
    dtype="boolean"
)

In [ ]:
h5_path_top = project_root + "/data/processed/amy_processeddata_top.h5"

with pd.HDFStore(h5_path_top, mode="w") as store:

    print('top')
    dissim_mtx = core.create_dissim_amy(raw_top, sub_no=-1, unique_words=unique_words)
    dist = core.cal_dissim2dist(dissim_mtx)
    store.put("/dissim/top", dissim_mtx, format="fixed")

    for i_t, t in enumerate(t_all):
        sim = core.cal_dist2sim(dist, t)
        genmag = core.cal_sim2genmag(sim)
        genmag_top.iloc[0, i_t] = genmag

        spr = core.cal_dist2spread(dist, t)
        spr_top.iloc[0, i_t] = spr

        A = sim.to_numpy()
        is_pd = core.check_positivedefinite(A)
        positive_definite_t_top.iloc[0, i_t] = bool(is_pd)

h5_path_bot = project_root + "/data/processed/amy_processeddata_bot.h5"
with pd.HDFStore(h5_path_bot, mode="w") as store:
    
    print('bottom')
    dissim_mtx = core.create_dissim_amy(raw_bottom, sub_no=-1, unique_words=unique_words)
    dist = core.cal_dissim2dist(dissim_mtx)
    store.put("/dissim/bottom", dissim_mtx, format="fixed")

    for i_t, t in enumerate(t_all):
        sim = core.cal_dist2sim(dist, t)
        genmag = core.cal_sim2genmag(sim)
        genmag_bot.iloc[0, i_t] = genmag

        spr = core.cal_dist2spread(dist, t)
        spr_bot.iloc[0, i_t] = spr

        A = sim.to_numpy()
        is_pd = core.check_positivedefinite(A)
        positive_definite_t_bot.iloc[0, i_t] = bool(is_pd)

In [ ]:
if genmag_top.columns.is_unique is False or spr_top.columns.is_unique is False:
    raise ValueError("Keys of columns are not unique. Recommend checking t setting.")


positive_definite_df_top = positive_definite_t_top.astype("bool")
positive_definite_df_bot = positive_definite_t_bot.astype("bool")
with pd.HDFStore(h5_path_top, mode="a") as store:
    store.put("/metrics/genmag", genmag_top, format="fixed")
    store.put("/metrics/spread", spr_top, format="fixed")
    store.put("/metrics/positivedefinite", positive_definite_df_top, format="fixed")

with pd.HDFStore(h5_path_bot, mode="a") as store:
    store.put("/metrics/genmag", genmag_bot, format="fixed")
    store.put("/metrics/spread", spr_bot, format="fixed")
    store.put("/metrics/positivedefinite", positive_definite_df_bot, format="fixed")

In [ ]:
t_sorted, y_1 = viz.prepare_metric_df(genmag_top)
t_sorted, y_2 = viz.prepare_metric_df(genmag_bot)
y_1 = np.squeeze(y_1)
y_2 = np.squeeze(y_2)

In [ ]:
fig, ax = plt.subplots(figsize=[9, 5])

ax.plot(t_sorted, y_1, label="10-least depressed")
ax.plot(t_sorted, y_2, label="10-most depressed")

ax.set_xscale('log')
ax.set_ylim([-1, 24])
ax.set_xlabel("Scale parameter t", fontsize=14)
ax.set_ylabel("Generalized magnitude", fontsize=14)
ax.legend()

fig.tight_layout()
plt.show()